$\dot{\psi} = H[\psi], \psi(x) = f(x, \theta) \implies {\ddot{\theta}} = \sum_x\alpha\nabla_{\theta}{f}^T(H[f] - \nabla_{\theta}{f}\dot{\theta})$

$\dot{\theta} = \int{(\nabla_{\theta}{f}\nabla_{\theta}{f}^T)^{-1}\nabla_{\theta}{f}^TH[f]dx}$

In [ ]:
import jax
import jax.numpy as jnp
import diffrax
import plotly.graph_objects as go

KEY = jax.random.key(seed = 43)

In [ ]:
def get_d2w_d2t(foo, operator, points_sampler, pp_k = 10):
    @jax.jit
    def compute_jt_epsilon_minus_jv(p, v_vec, eps, x):
        _, jv = jax.jvp(lambda param: foo(x, param), (p,), (v_vec,))
        residual = eps - jv
        f_val, vjp_fn = jax.vjp(lambda param: foo(x, param), p)
        jt_residual = vjp_fn(residual)[0]
        
        return jt_residual
    eps = operator(foo)
    def d2w_dt(x, Q):
        p, pp = Q[0], Q[1]
        target = eps(x, p)
        a = compute_jt_epsilon_minus_jv(p, pp, target, x)
        return a

    def final_dw_dt(Q):
        p, pp = Q[0], Q[1]
        x = points_sampler()
        q = jax.vmap(lambda x, q:d2w_dt(x, q), in_axes=(0, None))(x, Q)
        a = jax.tree_util.tree_map(lambda a:pp_k*jnp.mean(a, 0), q)
        return (pp, a)

    return final_dw_dt

---

In [ ]:
def point(x, p, v, u, w):
    # Вычисляем квадрат евклидова расстояния
    # jnp.sum((x - p)**2) превращает вектор в скаляр
    r = x - p
    d = jnp.sum((v*20*r)**2)
    #k = 20 / (jnp.sum((v*20*r)**2) + 1)
    k = jnp.exp(-d/(2*u**2))/2
    return jnp.array([v * k * jnp.sin(50 * w @ r), v * k * (jnp.sum(w)) * jnp.cos(50 * w @ r)])

def foo(x, pa):
    # Извлекаем параметры
    p = pa["p"]
    w = pa["w"]
    v = pa["v"]
    u = pa["u"]
    
    # Векторизуем point по p, v и u (оси 0), x остается константой (None)
    # vmapped_point будет возвращать массив формы (NUMPARAMS, 2)
    vmapped_point = jax.vmap(point, in_axes=(None, 0, 0, 0, 0))
    
    results = vmapped_point(x, p, v, u, w)
    
    # Суммируем по оси 0, чтобы получить итоговый вектор формы (2,)
    # Если вам нужен скаляр, оставьте просто jnp.sum(results)
    return jnp.mean(results, axis=0)

---

In [ ]:
def uniform_sampler(min_v, max_v, shape):
    def sampler():
        return jax.random.uniform(KEY, shape, jnp.float32, min_v, max_v)
    return sampler

In [ ]:
def operator(foo):
    k = 0.9
    hess = jax.hessian(lambda x, p:foo(x, p)[0])
    def laplacian_hessian(x, p):
        return jnp.trace(hess(x, p))

    def eps(x, p):
        L = laplacian_hessian(x, p)
        return jnp.stack([foo(x, p)[1], k*L])

    return eps

NUMPARAMS = 100
DIMS = 2
w0 = {
    "p":jax.random.uniform(KEY, (NUMPARAMS, DIMS), minval = -1, maxval=1),
    "w":jax.random.normal(KEY, (NUMPARAMS, DIMS))*5.0,
    "v":jax.random.normal(KEY, (NUMPARAMS))*5.0,
    "u":jax.random.normal(KEY, (NUMPARAMS))*5.0,
}
dw_dt0 = jax.tree_util.tree_map(jnp.zeros_like, w0)
Q = (w0, dw_dt0)
sampler = uniform_sampler(-1, 1, (1000, DIMS))
dQ_dt = get_d2w_d2t(foo, operator, sampler)

In [ ]:
T = 1
solver = diffrax.Euler()
term = diffrax.ODETerm(lambda t, y, args: dQ_dt(y))
saveat = diffrax.SaveAt(ts=jnp.linspace(0, T, 100),  t0 = True)
sol = diffrax.diffeqsolve(term, solver, t0=0, t1=T, dt0=0.05, y0=Q, saveat=saveat)

# Visualization

In [ ]:
def make_x_grid(xmin=-1.0, xmax=1.0, ymin=-1.0, ymax=1.0, nx=50, ny=50):
    xs = jnp.linspace(xmin, xmax, nx)
    ys = jnp.linspace(ymin, ymax, ny)
    xx, yy = jnp.meshgrid(xs, ys, indexing="xy")
    return jnp.stack([xx, yy], axis=-1)

def foo_grid(p, x_grid, foo):
    flat_x = x_grid.reshape(-1, DIMS)
    vals = jax.vmap(lambda x: foo(x, p))(flat_x)
    return vals.reshape(x_grid.shape[:-1])

In [ ]:
def make_p_evolution(p_evolution):
    """
    Преобразует эволюцию параметров из объекта Diffrax или кортежей 
    в плоский список состояний по шагам времени.
    """
    if hasattr(p_evolution, "ys"):
        p_evolution = [tuple(arr[i] for arr in p_evolution.ys) for i in range(p_evolution.ys[0].shape[0])]
    elif isinstance(p_evolution, tuple) and p_evolution and getattr(p_evolution[0], "ndim", 0) > 1:
        p_evolution = [tuple(arr[i] for arr in p_evolution) for i in range(p_evolution[0].shape[0])]
    return p_evolution

def plot_foo_timeline(x_grid, p_evolution, foo, title="foo evolution"):
    """
    Строит анимированный квадратный тепловой график (Heatmap) в Plotly,
    отображающий эволюцию системы по шагам времени.
    """
    #p_evolution = make_p_evolution(p_evolution)
    x = x_grid[0, :, 0].tolist()
    y = x_grid[:, 0, 1].tolist()

    # foo_grid должна быть определена в вашей области видимости
    # Создаем список индексов
    # indices = jnp.arange(len(p_evolution.ts))

    # Функция, которая вырезает одно состояние по индексу
    get_state = lambda i: jax.tree_util.tree_map(lambda leaf: leaf[i], p_evolution.ys[0])

    # Если нужно именно для цикла:
    state_at_t = [get_state(i) for i in range(len(p_evolution.ts))]
    z_list = [foo_grid(p, x_grid, foo) for p in state_at_t]
    zmin = float(min(jnp.min(z) for z in z_list))
    zmax = float(max(jnp.max(z) for z in z_list))

    def heatmap(z):
        return go.Heatmap(
            z=z.tolist(),
            x=x,
            y=y,
            colorscale="Viridis",
            zmin=zmin,
            zmax=zmax,
            colorbar=dict(title="foo")
        )

    frames = [
        go.Frame(
            data=[heatmap(z)],
            name=str(i),
            layout=go.Layout(title_text=f"{title} — step {i}")
        )
        for i, z in enumerate(z_list)
    ]

    fig = go.Figure(
        data=[heatmap(z_list[0])],
        frames=frames,
        layout=go.Layout(
            title=title,
            # Фиксируем физические размеры контейнера (в пикселях)
            width=650,
            height=650,
            
            # Настраиваем оси так, чтобы график не растягивался
            xaxis=dict(
                title="x",
                scaleanchor="y",  # Масштаб оси X жестко привязывается к оси Y
                scaleratio=1      # Соотношение масштабов физических пикселей 1:1
            ),
            yaxis=dict(
                title="y"
            ),
            updatemenus=[
                dict(
                    type="buttons",
                    showactive=False,
                    y=1.1,
                    x=1.05,
                    xanchor="right",
                    yanchor="top",
                    buttons=[
                        dict(
                            label="Play",
                            method="animate",
                            args=[
                                None,
                                dict(
                                    frame=dict(duration=250, redraw=True),
                                    transition=dict(duration=0),
                                    fromcurrent=True,
                                    mode="immediate",
                                ),
                            ],
                        ),
                        dict(
                            label="Pause",
                            method="animate",
                            args=[
                                [None],
                                dict(frame=dict(duration=0, redraw=False), mode="immediate"),
                            ],
                        ),
                    ],
                )
            ],
            sliders=[
                dict(
                    active=0,
                    currentvalue=dict(prefix="step: "),
                    pad=dict(t=50),
                    steps=[
                        dict(
                            label=str(i),
                            method="animate",
                            args=[
                                [str(i)],
                                dict(frame=dict(duration=0, redraw=True), transition=dict(duration=0), mode="immediate"),
                            ],
                        )
                        for i in range(len(frames))
                    ],
                )
            ],
        ),
    )

    return fig

In [ ]:
s = 2
x_grid = make_x_grid(-s, s, -s, s, 100, 100)
plot_foo_timeline(x_grid, sol, lambda x, p:foo(x, p)[0])

In [ ]:
Q = jax.tree_util.tree_map(lambda x:jnp.mean(jnp.abs(x), -1), sol.ys[1])
import matplotlib.pyplot as plt

plt.plot(Q['p'])
plt.show()